# 06 Deep Learning Pipeline

Multi-branch architecture: video ROI/frame clips, BVP sequence, EDA/TEMP/ACC physiology sequence, engineered static features, temporal CNN fusion, and three output heads.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import ARCHIVE_DIR, ALIGNED_DIR, FEATURES_DIR, MODELS_DIR
from src.data.archive_scanner import scan_archive
from src.data.alignment import save_aligned_trial
from src.features.build_features import build_feature_table, save_feature_table
from src.models.datasets import AlignedWindowDataset, add_window_targets, create_window_index, fit_static_scaler, fit_target_scalers, split_window_index
from src.models.dl_models import DLModelConfig, MultimodalPhysiologyNet, count_parameters
from src.models.dl_training import fit_multitask_model, save_checkpoint


## 1. Ensure aligned and feature outputs exist

In [ ]:
records = scan_archive(ARCHIVE_DIR)
if not list(ALIGNED_DIR.glob('*_aligned_1hz.csv')):
    for record in records:
        save_aligned_trial(record, ALIGNED_DIR, hz=1.0)

features_path = FEATURES_DIR / 'trial_features.csv'
if not features_path.exists():
    save_feature_table(ARCHIVE_DIR, features_path)
features = pd.read_csv(features_path)
features.head()


## 2. Build supervised aligned windows

In [ ]:
window_index = create_window_index(
    ALIGNED_DIR,
    archive_records=records,
    window_seconds=64,
    stride_seconds=32,
    sample_rate_hz=1.0,
)
windows = add_window_targets(window_index, features)
windows_path = FEATURES_DIR / 'dl_windows.csv'
windows.to_csv(windows_path, index=False)
windows.head(), windows.shape


## 3. Inspect dataset tensors

In [ ]:
train_windows, val_windows = split_window_index(windows, test_size=0.2)
static_scaler = fit_static_scaler(features)
target_scalers = fit_target_scalers(train_windows)

# Keep use_video=False for fast debugging. Set True when you want the video branch to read clips from MOV files.
train_ds = AlignedWindowDataset(train_windows, feature_table=features, sequence_length=64, video_frames=16, frame_size=64, use_video=False, static_scaler=static_scaler, target_scalers=target_scalers)
val_ds = AlignedWindowDataset(val_windows, feature_table=features, sequence_length=64, video_frames=16, frame_size=64, use_video=False, static_scaler=static_scaler, target_scalers=target_scalers)

batch = next(iter(DataLoader(train_ds, batch_size=2, shuffle=True)))
{key: tuple(value.shape) for key, value in batch.items() if torch.is_tensor(value)}


## 4. Build model and run one forward pass

In [ ]:
config = DLModelConfig(
    video_channels=batch['video'].shape[1],
    bvp_channels=batch['bvp'].shape[1],
    physiology_channels=batch['physiology'].shape[1],
    static_dim=batch['static'].shape[1],
    hidden_dim=96,
    temporal_dim=96,
)
model = MultimodalPhysiologyNet(config)
outputs = model(batch['video'], batch['bvp'], batch['physiology'], batch['static'])
count_parameters(model), {key: tuple(value.shape) for key, value in outputs.items()}


## 5. Tiny debug training run

In [ ]:
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

history = fit_multitask_model(model, train_loader, val_loader, epochs=10, lr=1e-3, device='cpu')
checkpoint_path = save_checkpoint(model, MODELS_DIR / 'multimodal_physio_net.pt', config=config.__dict__, history=history, static_scaler=static_scaler, target_scalers=target_scalers)
pd.DataFrame(history), checkpoint_path
